# Furniche: Photo to 3D on a free GPU

Runs [Tencent Hunyuan3D-2](https://github.com/Tencent-Hunyuan/Hunyuan3D-2) on a free cloud GPU so
Furniche's **Photo to 3D** builds textured models with no daily limits. It cuts the piece out of the
photo, removes the photo's lighting and reflections, then builds and paints the mesh: about a minute
per piece on a free T4.

**Kaggle (recommended: 2 × T4, ~29 GB RAM, 30 GPU hours a week)**
1. kaggle.com → Create → New Notebook → File → Import Notebook → upload this file.
2. Settings (right panel): **Accelerator → GPU T4 x2**, **Internet → On** (needs a phone-verified account).
3. Run all. The first run installs and downloads the models (about 15–25 minutes), then makes a test model.
4. Copy the two lines the last cell prints into Furniche's `.env.local` and restart `npm run dev`.
5. Leave the notebook running during the demo. If the session ends, run all again and paste the new lines.

**Colab** works too (Runtime → Change runtime type → **T4 GPU**), but it has less memory (12 GB RAM) and may run out.

Hunyuan3D-2 is under Tencent's non-commercial community license (and not licensed in the EU, UK or South Korea): fine for a hackathon demo.

In [ ]:
# 1. Install Hunyuan3D-2 (about 10 minutes the first time).
import os, subprocess, sys
BASE = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
os.chdir(BASE)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout
print(gpu or "No GPU found: turn on a GPU accelerator in the notebook settings, then run again.")
if not os.path.exists("Hunyuan3D-2"):
    !git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd Hunyuan3D-2
# Versions from the model's release (spring 2025): newer diffusers releases change APIs it relies on.
!pip install -q -r requirements.txt "diffusers==0.33.1" "transformers==4.51.3" "huggingface_hub<1" fastapi uvicorn rembg onnxruntime
!pip install -q -e .
!cd hy3dgen/texgen/custom_rasterizer && python3 setup.py install > /dev/null
!cd hy3dgen/texgen/differentiable_renderer && python3 setup.py install > /dev/null
print("Installed.")

In [ ]:
%%writefile photo3d_server.py
# Furniche Photo to 3D server: Tencent Hunyuan3D-2 (shape + texture) behind a small job API.
# Every endpoint but /health needs "Authorization: Bearer <PHOTO3D_TOKEN>".
#   GET  /health                               -> {"ok": true, "gpu": "..."}
#   POST /cutout  (image bytes)                 -> PNG of the piece on a transparent background
#   POST /jobs?quality=quick|detail (image)     -> {"id": "..."}
#   GET  /jobs/<id>  -> 202 {"state": "queued"|"running", "ahead": n} | 200 GLB | 500 {"error": "..."}
# Builds run one at a time on the GPU. A build is a job you poll, so no request outlasts a tunnel's timeout.
import io, os, queue, tempfile, threading, time, traceback, uuid

import torch
from fastapi import FastAPI, HTTPException, Request
from fastapi.concurrency import run_in_threadpool
from fastapi.responses import JSONResponse, Response
from PIL import Image
from rembg import new_session, remove

from hy3dgen.shapegen import DegenerateFaceRemover, FaceReducer, FloaterRemover, Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.texgen import Hunyuan3DPaintPipeline

TOKEN = os.environ["PHOTO3D_TOKEN"]
QUALITY = {
    "quick": {"octree_resolution": 256, "faces": 40_000},
    "detail": {"octree_resolution": 384, "faces": 100_000},
}

cutter = new_session("isnet-general-use")
shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2", subfolder="hunyuan3d-dit-v2-0-turbo", use_safetensors=True)
shape.enable_flashvdm(mc_algo="mc")
paint = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2", subfolder="hunyuan3d-paint-v2-0-turbo")
paint.enable_model_cpu_offload()  # with the shape model also on the GPU, a 16 GB T4 needs this

app = FastAPI()
jobs = {}
pending = queue.Queue()


def authorised(req: Request):
    if req.headers.get("authorization") != f"Bearer {TOKEN}":
        raise HTTPException(status_code=401, detail="bad token")


def cut_out(data: bytes) -> Image.Image:
    img = Image.open(io.BytesIO(data)).convert("RGBA")
    if img.getchannel("A").getextrema()[0] == 255:  # an ordinary photo: find the piece
        img = remove(img, session=cutter)
    return img


def build(img: Image.Image, quality: str) -> bytes:
    q = QUALITY[quality]
    with torch.inference_mode():
        mesh = shape(
            image=img,
            num_inference_steps=5,  # the turbo model is distilled for 5 steps
            octree_resolution=q["octree_resolution"],
            num_chunks=20000,
            generator=torch.Generator("cuda").manual_seed(1234),
            output_type="trimesh",
        )[0]
        mesh = FloaterRemover()(mesh)
        mesh = DegenerateFaceRemover()(mesh)
        mesh = FaceReducer()(mesh, max_facenum=q["faces"])
        mesh = paint(mesh, image=img)  # removes the photo's lighting and reflections, then paints
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "model.glb")
        mesh.export(path)
        with open(path, "rb") as f:
            return f.read()


def worker():
    while True:
        job_id, img, quality = pending.get()
        job = jobs.get(job_id)
        if job is None:
            continue
        job["state"] = "running"
        started = time.time()
        try:
            job["glb"] = build(img, quality)
            job["state"] = "done"
        except Exception as e:  # tell the app instead of dying
            traceback.print_exc()
            job["error"] = f"{type(e).__name__}: {e}"[:300]
            job["state"] = "failed"
        finally:
            job["seconds"] = round(time.time() - started, 1)
            print(f"job {job_id[:8]} {quality}: {job['state']} in {job['seconds']} s", flush=True)
            torch.cuda.empty_cache()


threading.Thread(target=worker, daemon=True).start()


@app.get("/health")
def health():
    return {"ok": True, "gpu": torch.cuda.get_device_name(0), "queued": pending.qsize()}


@app.post("/cutout")
async def cutout(req: Request):
    authorised(req)
    img = await run_in_threadpool(cut_out, await req.body())
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return Response(buf.getvalue(), media_type="image/png")


@app.post("/jobs")
async def submit(req: Request, quality: str = "detail"):
    authorised(req)
    img = await run_in_threadpool(cut_out, await req.body())
    for old in [k for k, j in list(jobs.items()) if time.time() - j["at"] > 3600]:
        jobs.pop(old, None)  # forget results nobody collected within an hour
    job_id = uuid.uuid4().hex
    jobs[job_id] = {"state": "queued", "at": time.time()}
    pending.put((job_id, img, quality if quality in QUALITY else "detail"))
    return {"id": job_id}


@app.get("/jobs/{job_id}")
def status(job_id: str, req: Request):
    authorised(req)
    job = jobs.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="unknown job")
    if job["state"] == "done":
        return Response(job["glb"], media_type="model/gltf-binary", headers={"X-Build-Seconds": str(job["seconds"])})
    if job["state"] == "failed":
        return JSONResponse({"state": "failed", "error": job["error"]}, status_code=500)
    ahead = sum(1 for j in list(jobs.values()) if j["state"] == "queued" and j["at"] < job["at"])
    return JSONResponse({"state": job["state"], "ahead": ahead}, status_code=202)

In [ ]:
# 2. Start the server and make one test model (downloads about 15 GB of models the first time).
import json, os, secrets, subprocess, sys, time, urllib.error, urllib.request
os.environ["PHOTO3D_TOKEN"] = TOKEN = os.environ.get("PHOTO3D_TOKEN") or secrets.token_urlsafe(24)
if "server" in globals() and server.poll() is None:
    server.terminate()
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "photo3d_server:app", "--host", "127.0.0.1", "--port", "8000"],
                          stdout=open("server.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
started = time.time()
while True:
    time.sleep(5)
    if server.poll() is not None:
        print(open("server.log").read()[-4000:])
        raise SystemExit("The server stopped while loading: see the log above.")
    try:
        print(json.load(urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3)))
        break
    except Exception:
        print(f"\rLoading models… {time.time() - started:.0f} s", end="")

def call(path, data=None):
    req = urllib.request.Request(f"http://127.0.0.1:8000{path}", data=data, headers={"Authorization": f"Bearer {TOKEN}"})
    try:
        with urllib.request.urlopen(req, timeout=600) as res:
            return res.status, res.read()
    except urllib.error.HTTPError as e:
        return e.code, e.read()

_, body = call("/jobs?quality=quick", open("assets/demo.png", "rb").read())
job = json.loads(body)["id"]
started = time.time()
while True:
    time.sleep(2)
    code, body = call(f"/jobs/{job}")
    if code != 202:
        break
if code != 200:
    print(open("server.log").read()[-4000:])
    raise SystemExit(f"The test model failed: {body[:300]!r}")
print(f"Test model OK: {len(body) / 1e6:.1f} MB in {time.time() - started:.0f} s")

In [ ]:
# 3. Open a public link (Cloudflare quick tunnel: free, no account) and print the lines for Furniche.
import re, subprocess, threading
if not os.path.exists("cloudflared"):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
if "tunnel" in globals() and tunnel.poll() is None:
    tunnel.terminate()
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        url = found.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # keep the pipe drained
print("Add these two lines to Furniche's .env.local, then restart npm run dev:\n")
print(f"PHOTO3D_GPU_URL={url}")
print(f"PHOTO3D_GPU_TOKEN={TOKEN}")

In [ ]:
# 4. Keep this running during the demo. It shows each build as it happens.
import time
seen = 0
while True:
    log = open("server.log").read()
    print(log[seen:], end="")
    seen = len(log)
    time.sleep(30)